# Lab 2.4 &mdash; Branch, Score, Prune &mdash; and When to Stop Reflecting

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 1 &middot; Module 2 &mdash; Agentic Planning &amp; Reasoning**

### What you'll do
- Generate three candidate resolutions concurrently with <code>RunnableParallel</code>
- Write the scorer that decides which one survives &mdash; and see it pick the trap
- Put a stop condition on a reflection loop before it runs out of your budget

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **The thread.** All five Module 2 labs work one case: an internal employee help desk.
> The rules are ordinary on purpose &mdash; the only new thing here is how the agent reasons.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-2-04")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------ the case file (synthetic, self-contained)
# An internal employee help desk. Ordinary rules on purpose: the only new thing in these five
# labs is LangChain. Nothing here is real data and nothing leaves this notebook.

REQUESTS = {
    "EHD-7001": {"who": "Priya Nair",   "category": "access",   "urgency": "high",
                 "wants": "reset",
                 "text": "Locked out of the payroll portal after the password reset."},
    "EHD-7002": {"who": "Rahul Menon",  "category": "hardware", "urgency": "high",
                 "wants": "replacement",
                 "text": "Laptop battery has swollen and the case is bulging."},
    "EHD-7003": {"who": "Anita Sharma", "category": "software", "urgency": "low",
                 "wants": "licence",
                 "text": "Need a licence for the diagramming tool, about 180 USD a year."},
    "EHD-7004": {"who": "Vikram Rao",   "category": "access",   "urgency": "medium",
                 "wants": "admin-rights",
                 "text": "Please give me admin rights on the finance reporting system."},
    "EHD-7005": {"who": "Priya Nair",   "category": "hardware", "urgency": "low",
                 "wants": "replacement",
                 "text": "Second monitor flickers every few minutes."},
}

# The handbook, one entry per category. Every judgement in this module comes from these.
HANDBOOK = {
    "access":   "Verify identity, then reset. The help desk NEVER grants elevated or admin "
                "rights -- route those to Identity and Access Management.",
    "hardware": "Replace under warranty. A swollen battery is a safety issue: stop use "
                "immediately and replace the same day, whatever urgency the employee set.",
    "software": "Licences over 100 USD per year need the cost-centre owner's approval first.",
}

SLA_HOURS = {"high": 4, "medium": 24, "low": 72}
ROUTE_OUT = {"admin-rights"}     # what the help desk must hand to another team, never do itself

print(f"{len(REQUESTS)} help desk requests, {len(HANDBOOK)} handbook entries loaded")

## Concept

Two ways of spending extra calls to get a better answer.

**Branch and prune:** generate several candidates, score them, keep one. `RunnableParallel` runs
them at the same time, so three candidates cost one round trip of latency rather than three.

**Reflection:** draft, criticise, revise. Improves the answer for a round or two, and then stops
paying while continuing to cost.

Both look like they are about generating. Neither is. Branching is about the **scorer**, and
reflection is about the **stop condition**, and those are the two things this lab makes you write.

The case is EHD-7004: Vikram Rao wants admin rights on the finance reporting system.
`wants` is `"admin-rights"`, which is in `ROUTE_OUT` &mdash; the help desk must hand it to another
team, never do it itself. So one of the three candidates is going to sound extremely helpful and
be exactly wrong.

## Section 1 &mdash; Three branches at once, and the scorer that judges them

`RunnableParallel(fastest=..., handbook=..., safest=...)` is one runnable holding three chains.
Invoke it once with one input and all three run concurrently; you get a dict back, keyed by name.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda

ANGLES = {
    "fastest":  "Get this employee working again today, with the fewest hops.",
    "handbook": "Answer strictly from the handbook, quoting its operative words.",
    "safest":   "Say what must NOT happen here, and who owns the decision instead.",
}


def branches_runnable(model):
    """One runnable, three chains, one invoke -- the branches run concurrently."""
    def chain(angle):
        return ChatPromptTemplate.from_messages([
            ("system", "You are an employee help desk analyst. " + angle + " Two sentences."),
            ("human", "REQUEST: {request}\nHANDBOOK: {handbook}\n\nWhat happens next?"),
        ]) | model | StrOutputParser()
    return RunnableParallel(**{name: chain(angle) for name, angle in ANGLES.items()})


def score_branch(text: str) -> int:
    """Score one candidate resolution of EHD-7004, out of 3."""
    t = text.lower()
    grants = any(p in t for p in ("grant admin", "grant the admin", "grant elevated",
                                  "grant him admin", "provision admin", "add him to the admin"))
    routes = "identity and access" in t or "iam" in t

    rewards_resolution = 3 if grants else (2 if routes else 1)   # "is the employee sorted?"
    rewards_handbook = 0 if grants else (3 if routes else 1)     # "did we follow the handbook?"

    return BLANK
    # TODO ^ EHD-7004 wants "admin-rights", and that is in ROUTE_OUT. One of these two scorers
    #        can reward a branch that grants them. Which one must you never ship?

In [ ]:
# --- Self-check: Section 1   (a real RunnableParallel, really invoked -- with no model in it)
CANDIDATES = {
    "fastest":  "Grant admin rights on finance reporting to Vikram Rao today so he is unblocked.",
    "handbook": "The help desk never grants elevated rights. Route this to Identity and Access "
                "Management, who own the entitlement.",
    "safest":   "Do not change any entitlement here. Identity and Access Management decides.",
}

ECHO = RunnableLambda(lambda pv: pv.to_string())      # stands in for the model, offline

def parallel_run():
    return branches_runnable(ECHO).invoke({"request": "EHD-7004", "handbook": HANDBOOK["access"]})

check("one invoke drives all three branches, each with its own framing",
      lambda: set(parallel_run()) == set(ANGLES) and len(set(parallel_run().values())) == 3,
      "three copies of one prompt is not a tree, it is one answer billed three times")
check("the branch that grants admin rights scores zero",
      lambda: score_branch(CANDIDATES["fastest"]) == 0,
      "EHD-7004 is a ROUTE_OUT case: resolving it here is a breach, however fast it is")
check("the routing branches score full marks, and pruning keeps one of them",
      lambda: score_branch(CANDIDATES["handbook"]) == 3
              and max(CANDIDATES, key=lambda k: score_branch(CANDIDATES[k])) != "fastest")
score()

## Section 2 &mdash; Reflection, and the knee

Draft, criticise, revise. The first revision usually earns its call. The second sometimes does. By
the fourth the critic is inventing work, and you are paying two calls a round for a rewording.

That curve has a knee, and a reflection loop with no stop condition is Module 2's most expensive
failure &mdash; it does not crash, it just bills.

So the loop is three lines. What you have to decide is when it ends.

In [ ]:
MAX_ROUNDS = 3


def should_stop(round_no: int, critique: str, last_score: int, this_score: int) -> bool:
    """Called after each revision. True = stop reflecting."""
    critic_found_nothing = critique.strip().lower().startswith("no change")
    out_of_rounds = round_no >= MAX_ROUNDS
    no_longer_paying = this_score <= last_score

    stop_when_clean = critic_found_nothing
    stop_on_any = critic_found_nothing or out_of_rounds or no_longer_paying

    return BLANK
    # TODO ^ a critic asked to find a fault will usually find one. Which of these can never run
    #        forever AND never keeps paying past the knee?


def reflect(draft: str, critique_of, revise, hard_cap: int = MAX_ROUNDS + 4) -> list:
    """[(round, text, score), ...]. hard_cap is a seatbelt, not the stop condition."""
    history = [(0, draft, score_branch(draft))]
    text = draft
    for n in range(1, hard_cap + 1):
        critique = critique_of(text)
        text = revise(text, critique)
        previous, now = history[-1][2], score_branch(text)
        history.append((n, text, now))
        if should_stop(n, critique, previous, now):
            break
    return history

In [ ]:
# --- Self-check: Section 2   (a canned critic and reviser -- deterministic, no model)
DRAFTS = [CANDIDATES["fastest"],
          "Do not grant anything. Send this to Identity and Access Management.",
          "Do not change entitlements. Identity and Access Management owns admin rights here."]

def canned_critique(text: str) -> str:
    return ("no change needed" if "identity and access" in text.lower()
            else "the handbook routes elevated rights to another team")

def canned_revise(text: str, critique: str) -> str:
    i = DRAFTS.index(text) if text in DRAFTS else len(DRAFTS) - 1
    return DRAFTS[min(i + 1, len(DRAFTS) - 1)]

def canned_history():
    return reflect(DRAFTS[0], canned_critique, canned_revise)


check("the stop rule fires when the critic runs out of things to say",
      lambda: should_stop(1, "no change needed", 2, 3) is True)
check("it also fires on the round budget, however talkative the critic",
      lambda: should_stop(MAX_ROUNDS, "one more nit", 1, 2) is True
              and should_stop(1, "one more nit", 1, 2) is False,
      "a critic asked for a fault will invent one, so 'until it is clean' alone never terminates")
check("round 1 earns its call (0 -> 3), and the loop then stops at the knee",
      lambda: canned_history()[1][2] == 3 and len(canned_history()) == 3,
      "round 2 scored the same as round 1; paying for round 3 is the failure this section prevents")
score()

## Run it for real

Three branches from one `invoke`, scored and pruned. Then three rounds of reflection with the
score printed each round, so you can see where the knee is on this case.

In [ ]:
def live_branches():
    r = REQUESTS["EHD-7004"]
    out = branches_runnable(get_llm()).invoke(
        {"request": json.dumps({"id": "EHD-7004", **r}), "handbook": HANDBOOK[r["category"]]})
    for name, text in out.items():
        print(f"[{score_branch(text)}] {name}: {' '.join(text.split())[:150]}")
    best = max(out, key=lambda k: score_branch(out[k]))
    print("\nkept:", best, "| pruned:", [k for k in out if k != best])

if llm_ready():
    guard(live_branches)

In [ ]:
def live_reflection():
    r = REQUESTS["EHD-7004"]
    ctx = f'REQUEST: {json.dumps({"id": "EHD-7004", **r})}\nHANDBOOK: {HANDBOOK[r["category"]]}'

    def critic(text):
        return ask(f"{ctx}\n\nDRAFT: {text}\n\nName the single worst way this draft departs "
                   "from the handbook, in one sentence. If it does not, reply exactly with: "
                   "no change needed")

    def reviser(text, critique):
        return ask(f"{ctx}\n\nDRAFT: {text}\nCRITIQUE: {critique}\n\nRewrite the draft in two "
                   "sentences, fixing only what the critique names. Output the draft only.")

    for n, text, sc in reflect(CANDIDATES["fastest"], critic, reviser):
        print(f"round {n}: score {sc} | {' '.join(text.split())[:120]}")

if llm_ready():
    guard(live_reflection)

### Read it

Look at the branch scores first. The `fastest` branch is fluent, confident and does the thing the
handbook says the help desk never does. Under a scorer that rewards "the employee is working
again", it wins. Nothing about the branching stopped that &mdash; the branching only produced the
candidate. **The scorer is the design decision.** Writing three prompts is the easy half.

Now the reflection column. The first round moves the score; after that the number stops moving
while the calls keep going out. That is the knee, and it is the whole reason `should_stop` exists.
A loop that runs "until the critic is happy" does not terminate, because a critic asked to find a
fault finds one &mdash; so the budget is not a fallback, it is the termination proof.

`hard_cap` in `reflect` is a seatbelt so a wrong stop rule shows up as a long run rather than a
hung notebook. Do not mistake it for a stop condition; it is what you are trying not to reach.

In [ ]:
score()

## Your turn

1. Change `score_branch` to return the other scorer and re-run the live branch cell. Which branch
   is kept now, and what would have happened to Vikram Rao's entitlements?
2. Set `MAX_ROUNDS` to 8 and run the live reflection again. Count the calls, then find the last
   round that changed the score.